In [2]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv('diabetes.csv')

In [4]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [5]:
 df.corr()['Outcome']

,Outcome
Pregnancies,0.221898
Glucose,0.466581
BloodPressure,0.065068
SkinThickness,0.074752
Insulin,0.130548
BMI,0.292695
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


In [6]:
X = df.iloc[:,:-1].values
y = df.iloc[:,-1].values

In [7]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X = scaler.fit_transform(X)


In [9]:
X.shape

(768, 8)

In [12]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=1)

In [13]:
import tensorflow
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

In [22]:
model = Sequential()

model.add(Dense(32,activation='relu',input_dim=8))
model.add(Dense(1,activation='sigmoid'))

model.compile(optimizer='Adam', loss='binary_crossentropy',metrics=['accuracy'])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [23]:
model.fit(X_train, y_train, batch_size=32, epochs=10, validation_data=(X_test,y_test))

Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.3290 - loss: 0.8126 - val_accuracy: 0.3312 - val_loss: 0.7530
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.3893 - loss: 0.7272 - val_accuracy: 0.5130 - val_loss: 0.6922
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6384 - loss: 0.6685 - val_accuracy: 0.6364 - val_loss: 0.6480
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6726 - loss: 0.6263 - val_accuracy: 0.7013 - val_loss: 0.6169
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6971 - loss: 0.5952 - val_accuracy: 0.7273 - val_loss: 0.5931
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7231 - loss: 0.5722 - val_accuracy: 0.7338 - val_loss: 0.5729
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7296 - loss: 0.5535 - val_accuracy: 0.7273 - val_loss: 0.5565
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7362 - loss: 0.5375 - val_accuracy: 0.7468 - val_loss

# Hyper Parameter Tuning Using Keras

 It is very tough to select hyperparameter manually, so we'll learn how to select them automatically using keras

1. How to select appropriate optimizer
2. No of nodes in a layer
3. How to select no. of layers
4. All in all one model

In [25]:
pip install -U keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 2.0 MB/s eta 0:00:00


In [26]:
import kerastuner as kt

/tmp/ipykernel_20191/1654478174.py:1: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  import kerastuner as kt


# Building function to choose OPTIMIZER

In [27]:
def build_model(hp):

  model = Sequential()

  model.add(Dense(32,activation='relu',input_dim=8))
  model.add(Dense(1,activation='sigmoid'))

  optimizer=hp.Choice('optimizer',values =['adam','sgd','rmsprop','adadelta'])

  model.compile(optimizer=optimizer, loss='binary_crossentropy',metrics=['accuracy'])

  return model

In [29]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=5
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [31]:
tuner.search(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

best_model = tuner.get_best_models(num_models=1)[0]

test_loss, test_acc = best_model.evaluate(X_test, y_test)
print("Test Accuracy:", test_acc)

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7792 - loss: 0.5244  
Test Accuracy: 0.7792207598686218


In [32]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'rmsprop'}

In [33]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [34]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [35]:
model.fit(X_train,y_train,batch_size=32,epochs=100,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step - accuracy: 0.7427 - loss: 0.5251 - val_accuracy: 0.7792 - val_loss: 0.5018
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7606 - loss: 0.5071 - val_accuracy: 0.7727 - val_loss: 0.4907
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.7622 - loss: 0.4964 - val_accuracy: 0.7727 - val_loss: 0.4841
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.7655 - loss: 0.4879 - val_accuracy: 0.7727 - val_loss: 0.4798
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.7704 - loss: 0.4817 - val_accuracy: 0.7792 - val_loss: 0.4765
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.7704 - loss: 0.4774 - val_accuracy: 0.7792 - val_loss: 0.4745
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.7720 - loss: 0.4729 - val_accuracy: 0.7857 - val_loss: 0.4722
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.7720 - loss: 0.4700 - val_accurac

# Building function to choose NO. OF NEURONS

In [48]:
def build_model_units(hp):

    model = Sequential()

    units = hp.Int('units', min_value=8, max_value=128, step=1)

    model.add(Dense(units, activation='relu', input_dim=8))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [49]:
tuner_units = kt.RandomSearch(
    build_model_units,
    objective='val_accuracy',
    max_trials=5,
    directory='tuner_dir',
    project_name='units_tuning'
)

Reloading Tuner from tuner_dir/units_tuning/tuner0.json


In [50]:
tuner_units.search(X_train, y_train, epochs=5, validation_data=(X_test, y_test))
tuner_units.get_best_hyperparameters()[0].values

{'units': 128}

In [51]:
tuner1.get_best_hyperparameters()[0].values

{'optimizer': 'rmsprop'}

# Tuner for NUMBER OF LAYERS

In [52]:
def build_model_layers(hp):

    model = Sequential()

    # Tune number of hidden layers (1–3)
    for i in range(hp.Int('num_layers', 1, 3)):
        if i == 0:
            model.add(Dense(32, activation='relu', input_dim=8))
        else:
            model.add(Dense(32, activation='relu'))

    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [53]:
tuner_layers = kt.RandomSearch(
    build_model_layers,
    objective='val_accuracy',
    max_trials=5,
    directory='tuner_dir',
    project_name='layers_tuning'
)

In [54]:
tuner_layers.search(X_train, y_train, epochs=5, validation_data=(X_test, y_test))
tuner_layers.get_best_hyperparameters()[0].values

Trial 3 Complete [00h 00m 03s]
val_accuracy: 0.8051947951316833

Best val_accuracy So Far: 0.8051947951316833
Total elapsed time: 00h 00m 13s


{'num_layers': 2}

# ALL 3 COMBINED TOGETHER

In [60]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import keras_tuner as kt

In [61]:
def build_model_all(hp):

    model = Sequential()

    # 🔹 Tune number of hidden layers (1–3)
    num_layers = hp.Int('num_layers', min_value=1, max_value=3)

    for i in range(num_layers):

        # 🔹 Tune neurons in each layer
        units = hp.Int(f'units_{i}', min_value=8, max_value=128, step=8)

        if i == 0:
            model.add(Dense(units, activation='relu', input_dim=8))
        else:
            model.add(Dense(units, activation='relu'))

    # 🔹 Output layer (fixed)
    model.add(Dense(1, activation='sigmoid'))

    # 🔹 Tune optimizer
    optimizer_choice = hp.Choice('optimizer', ['adam', 'rmsprop', 'sgd'])

    # 🔹 Tune learning rate
    lr = hp.Choice('learning_rate', [1e-2, 1e-3, 1e-4])

    if optimizer_choice == 'adam':
        optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
    elif optimizer_choice == 'rmsprop':
        optimizer = tf.keras.optimizers.RMSprop(learning_rate=lr)
    else:
        optimizer = tf.keras.optimizers.SGD(learning_rate=lr)

    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [62]:
tuner_all = kt.RandomSearch(
    build_model_all,
    objective='val_accuracy',
    max_trials=10,
    directory='tuner_dir',
    project_name='full_tuning'
)

In [63]:
tuner_all.search(
    X_train, y_train,
    epochs=10,
    validation_data=(X_test, y_test)
)

Trial 10 Complete [00h 00m 04s]
val_accuracy: 0.8051947951316833

Best val_accuracy So Far: 0.8246753215789795
Total elapsed time: 00h 00m 42s


In [64]:
best_hp = tuner_all.get_best_hyperparameters()[0]
print(best_hp.values)

{'num_layers': 2, 'units_0': 128, 'optimizer': 'adam', 'learning_rate': 0.001, 'units_1': 32, 'units_2': 24}


In [65]:
best_model = tuner_all.get_best_models(num_models=1)[0]

best_model.fit(X_train, y_train, epochs=20, validation_data=(X_test, y_test))

Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.7850 - loss: 0.4444 - val_accuracy: 0.8312 - val_loss: 0.4575
Epoch 2/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7899 - loss: 0.4325 - val_accuracy: 0.7987 - val_loss: 0.4587
Epoch 3/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7866 - loss: 0.4289 - val_accuracy: 0.8117 - val_loss: 0.4629
Epoch 4/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7866 - loss: 0.4259 - val_accuracy: 0.8052 - val_loss: 0.4586
Epoch 5/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7915 - loss: 0.4254 - val_accuracy: 0.8117 - val_loss: 0.4556
Epoch 6/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7915 - loss: 0.4234 - val_accuracy: 0.8247 - val_loss: 0.4558
Epoch 7/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8046 - loss: 0.4190 - val_accuracy: 0.8182 - val_loss: 0.4579
Epoch 8/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8046 - loss: 0.4146 - val_accuracy: 0.8182 - val_loss: 0.4584
E